## Final Data Preparation for Tableau

The dataset was enhanced with machine learning outputs:

- Prediction probability (likelihood of earning >50K)
- Predicted income classification
- Cluster labels (segmentation)
- Interpretable cluster names

This enriched dataset enables advanced analytics and interactive dashboarding in Tableau.


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/content/adult_cleaned.csv')
df.head()

,age,workclass,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,age_group
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0,Youth
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0,YoungAdult
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1,YoungAdult
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1,YoungAdult
4,18,Private,Some-college,10,Never-married,Prof-specialty,Own-child,White,Female,0,0,30,United-States,0,Youth


In [3]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

le = LabelEncoder()

for col in df_encoded.columns:
    if df_encoded[col].dtype == 'object':
        df_encoded[col] = le.fit_transform(df_encoded[col])

In [4]:
X = df_encoded.drop('income', axis=1)

In [5]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X, df_encoded['income'])

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

In [6]:
df_final = df.copy()

df_final['prediction_probability'] = model.predict_proba(X)[:, 1]
df_final['predicted_income'] = (df_final['prediction_probability'] >= 0.4).astype(int)

In [7]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
df_encoded['cluster'] = kmeans.fit_predict(X)

df_final['cluster'] = df_encoded['cluster']

In [8]:
cluster_names = {
    0: "Standard Workforce",
    1: "Elite High-Income Group",
    2: "Emerging High Earners"
}

df_final['cluster_name'] = df_final['cluster'].map(cluster_names)

In [9]:
df_final.to_csv('final_dataset.csv', index=False)

In [10]:
from google.colab import files
files.download('final_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
df_final.columns

Index(['age', 'workclass', 'education', 'education_num', 'marital_status',
       'occupation', 'relationship', 'race', 'sex', 'capital_gain',
       'capital_loss', 'hours_per_week', 'native_country', 'income',
       'age_group', 'prediction_probability', 'predicted_income', 'cluster',
       'cluster_name'],
      dtype='object')